# 03 Knowledge Graph: A-Share Sector Event Propagation

This notebook demonstrates the **SectorGraph** built from 31 SW Level-1 industries.
It shows:
1. Build and inspect the sector graph
2. Propagate key events (semiconductor policy, new energy policy, real estate easing)
3. Visualize the full sector graph as an interactive force-directed network (Plotly)
4. Top-10 affected sectors table for '半导体扩产' scenario
5. Bar chart: propagation scores for selected event

In [ ]:
import sys
import os
from pathlib import Path

repo_root = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
python_path = repo_root / 'python'
if str(python_path) not in sys.path:
    sys.path.insert(0, str(python_path))

print(f'Repo root: {repo_root}')

In [ ]:
import json
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px

from trade_py.knowledge_graph import (
    SectorGraph, SW, SW_NAMES_ZH,
    _EDGES, EVENT_SECTOR_MAPPING,
)

print('knowledge_graph module loaded')

## 1. Build and Inspect the Sector Graph

In [ ]:
graph = SectorGraph()
d = graph.to_dict()

print(f"Nodes (sectors): {len(d['nodes'])}")
print(f"Edges:           {len(d['edges'])}")
print(f"Event types:     {len(d['event_mappings'])}")
print()
print('Available event types:')
for evt in sorted(graph.available_events()):
    print(f'  {evt}')

In [ ]:
# Save graph JSON
graph_out = repo_root / 'data' / 'knowledge_graph' / 'sector_graph.json'
graph.save(graph_out)
print(f'Graph saved to: {graph_out}')

In [ ]:
# Edge breakdown by relation type
edges_df = pd.DataFrame(d['edges'])
rel_counts = edges_df['relation'].value_counts().reset_index()
rel_counts.columns = ['relation_type', 'count']
print(rel_counts.to_string(index=False))

## 2. Event Propagation Examples

In [ ]:
def show_propagation(event_type: str, max_hop: int = 2):
    results = graph.propagate_event(event_type, max_hop=max_hop)
    rows = []
    for r in results:
        rows.append({
            'sector_zh': r.sector_name,
            'score': round(r.score, 3),
            'hop': r.hop,
            'typical_days': r.typical_days,
            'relation': r.relation,
            'path': ' -> '.join(p.replace('SW_', '').replace(f'event:{event_type}', 'EVT')
                                for p in r.path),
        })
    df = pd.DataFrame(rows)
    pos = (df['score'] > 0).sum()
    neg = (df['score'] < 0).sum()
    print(f"\nEvent: {event_type}  |  {len(df)} sectors affected  "
          f"(+{pos} positive / -{neg} negative)")
    return df

In [ ]:
# semiconductor_policy propagation
df_semi = show_propagation('semiconductor_policy')
df_semi

In [ ]:
# new_energy_policy propagation
df_energy = show_propagation('new_energy_policy')
df_energy

In [ ]:
# real_estate_easing propagation
df_realestate = show_propagation('real_estate_easing')
df_realestate

## 3. Interactive Sector Network Graph (Force-Directed, Plotly)

In [ ]:
# Build NetworkX graph
G = nx.DiGraph()

for node in d['nodes']:
    G.add_node(node['id'], label=node['name_zh'], sw_code=node['sw_code'])

for edge in d['edges']:
    G.add_edge(
        edge['source'],
        edge['target'],
        relation=edge['relation'],
        weight=edge['weight'],
        direction=edge['direction'],
    )

print(f'NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Spring layout
import random
random.seed(42)
pos = nx.spring_layout(G, k=2.5, iterations=100, seed=42)

In [ ]:
# Color edges by relation type
RELATION_COLORS = {
    'upstream_supply':   '#1f77b4',  # blue
    'downstream_demand': '#ff7f0e',  # orange
    'policy_linkage':    '#2ca02c',  # green
    'fund_rotation':     '#9467bd',  # purple
    'macro_exposure':    '#8c564b',  # brown
    'competition':       '#d62728',  # red
}

fig = go.Figure()

# Draw edges grouped by relation type
edges_by_rel = {}
for src, tgt, data in G.edges(data=True):
    rel = data.get('relation', 'other')
    edges_by_rel.setdefault(rel, []).append((src, tgt, data))

for rel, edge_list in edges_by_rel.items():
    xe, ye = [], []
    for src, tgt, data in edge_list:
        x0, y0 = pos[src]
        x1, y1 = pos[tgt]
        xe += [x0, x1, None]
        ye += [y0, y1, None]
    color = RELATION_COLORS.get(rel, '#aaa')
    fig.add_trace(go.Scatter(
        x=xe, y=ye,
        mode='lines',
        line=dict(color=color, width=1.2),
        opacity=0.55,
        name=rel,
        legendgroup=rel,
        hoverinfo='none',
    ))

# Draw nodes
node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_labels = [G.nodes[n]['label'] for n in G.nodes()]
node_degrees = [G.degree(n) for n in G.nodes()]

fig.add_trace(go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    marker=dict(
        size=[8 + d * 1.5 for d in node_degrees],
        color=node_degrees,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Degree', thickness=12),
        line=dict(width=1, color='white'),
    ),
    text=node_labels,
    textposition='top center',
    textfont=dict(size=9),
    hovertext=[f"{G.nodes[n]['label']} ({n})" for n in G.nodes()],
    hoverinfo='text',
    name='Sector',
    showlegend=False,
))

fig.update_layout(
    title='A-Share Sector Knowledge Graph (SW Level-1)',
    showlegend=True,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=700,
    margin=dict(l=20, r=20, t=60, b=20),
    paper_bgcolor='#f8f9fa',
    plot_bgcolor='#f8f9fa',
)
fig.show()

## 4. Top-10 Affected Sectors: 半导体扩产 (Semiconductor Expansion)

In [ ]:
# 'semiconductor_policy' models the semiconductor expansion scenario
results_semi = graph.propagate_event('semiconductor_policy', max_hop=2)

top10 = results_semi[:10]
rows = []
for i, r in enumerate(top10, 1):
    path_str = ' -> '.join(
        p.replace('SW_', '').replace('event:semiconductor_policy', '半导体政策')
        for p in r.path
    )
    rows.append({
        '排名': i,
        '板块': r.sector_name,
        '传导分数': f'{r.score:+.3f}',
        '传导跳数': r.hop,
        '预期滞后(日)': r.typical_days,
        '关系类型': r.relation,
        '传导路径': path_str,
    })

top10_df = pd.DataFrame(rows)
print('半导体扩产政策 -> 传导影响 Top 10 板块')
print(top10_df.to_string(index=False))

## 5. Bar Chart: Propagation Scores by Sector

In [ ]:
def plot_propagation_scores(event_type: str, max_hop: int = 2, top_n: int = 20):
    results = graph.propagate_event(event_type, max_hop=max_hop)
    # Show top_n by abs score
    shown = results[:top_n]
    
    df = pd.DataFrame({
        'sector': [r.sector_name for r in shown],
        'score': [r.score for r in shown],
        'hop': [r.hop for r in shown],
    })
    df = df.sort_values('score')
    
    colors = ['#e84545' if s > 0 else '#2ba02b' for s in df['score']]
    
    fig = go.Figure(go.Bar(
        x=df['score'],
        y=df['sector'],
        orientation='h',
        marker_color=colors,
        text=[f"{s:+.2f}" for s in df['score']],
        textposition='outside',
        customdata=df['hop'],
        hovertemplate='%{y}<br>score: %{x:+.3f}<br>hop: %{customdata}<extra></extra>',
    ))
    fig.update_layout(
        title=f'Event: {event_type} — Sector Propagation Scores (top {top_n})',
        xaxis_title='Propagation Score',
        yaxis_title='Sector',
        height=max(400, len(df) * 28),
        xaxis=dict(zeroline=True, zerolinewidth=2, zerolinecolor='black'),
        margin=dict(l=120, r=80),
        showlegend=False,
    )
    fig.show()


plot_propagation_scores('semiconductor_policy')

In [ ]:
plot_propagation_scores('new_energy_policy')

In [ ]:
plot_propagation_scores('real_estate_easing')

## 6. Graph Reload Round-Trip Test

In [ ]:
# Load from saved JSON and verify results match
graph2 = SectorGraph.load(graph_out)
r1 = graph.propagate_event('semiconductor_policy')
r2 = graph2.propagate_event('semiconductor_policy')

assert len(r1) == len(r2), f'Length mismatch: {len(r1)} vs {len(r2)}'
for a, b in zip(r1, r2):
    assert abs(a.score - b.score) < 1e-4, f'Score mismatch for {a.sector_name}: {a.score} vs {b.score}'

print('Round-trip load/save: OK')
print(f'  {len(r1)} sectors propagated from semiconductor_policy')